# Gold Layer: Feature Engineering and Model Preparation

This notebook performs feature selection, creates the target variable, derives additional features, handles categorical encoding, and prepares the final dataset for machine learning models.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Set current date
CURRENT_DATE = pd.to_datetime('2026-04-13')

print("Loading joined dataset from silver layer...")

Loading joined dataset from silver layer...


In [2]:
# Load joined dataset
date_cols = ['agreement_end_date', 'case_creation_date', 'resolved_date', 'registered_date', 'expected_pull_date', 'agreement_start_date', 'agreement_end_date']
df = pd.read_csv('../../data/02_intermediate/joined_data.csv', parse_dates=date_cols)

print(f"Loaded dataset shape: {df.shape}")
print("Columns:", df.columns.tolist())

Loaded dataset shape: (715965, 40)
Columns: ['case_title', 'pull_van', 'new_van', 'van', 'number_of_contracts', 'machines', 'pull_type', 'case_type', 'risk', 'current_status', 'resolution_status', 'number_of_repair_cases', 'number_of_overdueservices', 'companysize', 'customer_tier', 'case_origin', 'case_creation_date', 'resolved_date', 'registered_date', 'expected_pull_date', 'account_number', 'company_sizing', 'branch', 'agreement_number', 'agreement_start_date', 'agreement_end_date', 'renewal_type', 'agreement_type', 'line_of_business', 'product_bob', 'fee_bob', 'total_bob', 'service_interval', 'unit_amount', 'billing_interval', 'billing_period', 'total_agreements', 'lost_agreements', 'active_agreements', 'churn_category']


In [3]:
def create_target_and_features(df: pd.DataFrame) -> pd.DataFrame:
    metrics = ['total_agreements', 'lost_agreements', 'active_agreements']
    if 'churn_category' in df.columns:
        metrics.append('churn_category')

    exclude_cols = ['account_number', 'customer_account_number', 'agreement_number'] + metrics
    feature_cols = [col for col in df.columns if col not in exclude_cols]

    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [col for col in feature_cols if col not in numeric_cols]

    agg_dict = {}
    for col in numeric_cols:
        agg_dict[col] = 'mean'
    for col in categorical_cols:
        agg_dict[col] = lambda x: x.mode(dropna=True).iloc[0] if len(x.mode(dropna=True)) > 0 else np.nan

    grouped = df.groupby('account_number', dropna=False).agg(agg_dict).reset_index()

    account_metrics = df.groupby('account_number', dropna=False).agg(
        total_agreements=('total_agreements', 'first'),
        lost_agreements=('lost_agreements', 'first'),
        active_agreements=('active_agreements', 'first')
    ).reset_index()
    grouped = grouped.merge(account_metrics, on='account_number', how='left')

    def label_churn(row):
        if row['active_agreements'] == 0:
            return 2
        if row['lost_agreements'] == 0:
            return 0
        return 1

    grouped['churn_category'] = grouped.apply(label_churn, axis=1)

    for col in categorical_cols:
        if col in grouped.columns:
            grouped[col] = pd.factorize(grouped[col].fillna('Unknown'))[0]

    final_cols = ['account_number']
    final_cols += ['total_agreements', 'lost_agreements', 'active_agreements', 'churn_category']
    final_cols += [col for col in grouped.columns if col not in final_cols]
    return grouped[final_cols].copy()

model_df = create_target_and_features(df)

os.makedirs('../../data/03_processed', exist_ok=True)
model_df.to_csv('../../data/03_processed/model_ready_data.csv', index=False)

print("Model-ready dataset saved to ../../data/03_processed/model_ready_data.csv")
print(f"Final dataset shape: {model_df.shape}")
print("Target distribution:")
print(model_df['churn_category'].value_counts())

Model-ready dataset saved to ../../data/03_processed/model_ready_data.csv
Final dataset shape: (21526, 39)
Target distribution:
churn_category
2    16233
0     4226
1     1067
Name: count, dtype: int64
